# 3 · Create controlled changes and measure detection

The source has no verified incident labels. We create our own changes, keep a log and ask whether the models recover them. These are controlled experiments, not proof of real incident detection.

In [ ]:
import json
import pandas as pd
import numpy as np
from IPython.display import display, Image
from traffic_anomaly.paths import ROOT, RAW, RESULTS, FIGURES, config
from traffic_anomaly.data import load_split
cfg = config()
print("Project configuration found:", (ROOT / "configs/study.json").is_file())
from traffic_anomaly.injection import inject
from traffic_anomaly.evaluation import metrics
from traffic_anomaly.models import load_models
models, thresholds = load_models()
validation = load_split("validation")

## Generate one repeatable experiment

A random seed is a saved number that lets us repeat the same random choices. Injection changes a copy. Spikes increase a count; drops reduce it; shifts alter a continuous block; flatlines repeat the first value. Blocks never cross missing-hour gaps.

In [ ]:
trial = inject(validation, "drop", "severe", .01, 101, cfg)
print("Actually changed hours:", int(trial.labels.sum()))
print("Original validation rows:", len(validation))
display(trial.changes.head())
display(trial.events.head())
assert np.array_equal(trial.labels.astype(bool), trial.frame.traffic_volume.to_numpy() != validation.traffic_volume.to_numpy())

## Compare the methods on identical changed records

Recall asks how many changed hours were found. Precision asks how many alerts correspond to our changes. A background alert is a flag on an unchanged hour; it may still be a real anomaly, so it is not a verified false alarm. Event recall and per-hour recall answer different questions.

In [ ]:
rows = []
for name, model in models.items():
    row = metrics(trial.labels, model.score(trial.frame), thresholds[name][str(cfg["primary_alert_rate"])], validation.date_time, trial.event_ids)
    rows.append({"method":name, **row})
display(pd.DataFrame(rows)[["method","n_injected","true_positive","false_negative","benchmark_false_positive","precision","recall","f1","average_precision","background_alert_rate"]])

## See a controlled change

The first changed position is chosen by the fixed seed, not selected because a detector performed well.

In [ ]:
import matplotlib.pyplot as plt
first = int(np.flatnonzero(trial.labels)[0])
a, b = max(0,first-24), first+25
fig, ax = plt.subplots(figsize=(9,3.5))
ax.plot(validation.date_time.iloc[a:b], validation.traffic_volume.iloc[a:b], label="Original")
ax.plot(trial.frame.date_time.iloc[a:b], trial.frame.traffic_volume.iloc[a:b], label="Changed copy", linestyle="--")
ax.set(ylabel="Vehicles per hour")
ax.legend()
fig.autofmt_xdate()
plt.show()

## Why one example is not enough

The complete experiment repeats seeds, anomaly recipes, alert thresholds and prevalence levels. Read `configs/study.json` to see every setting. After validation, `artifacts/frozen_protocol.json` records hashes so edits cannot silently reuse old test results.

Next: `04_read_the_results.ipynb`.